# InferGrid Phase 1: Scheduling Overhead Analysis

This notebook loads profiling and benchmark results from vLLM and SGLang,
then produces publication-quality visualizations for the scheduling overhead
analysis.

**Key questions:**
1. What fraction of inference time is CPU scheduling overhead?
2. How large is the throughput gap between vLLM and SGLang?
3. Where are the bottleneck functions (from flame graphs)?
4. What are the GPU utilization patterns under load?

In [ ]:
"""Setup — imports and configuration."""

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Use academic paper style
try:
    plt.style.use("seaborn-v0_8-paper")
except OSError:
    plt.style.use("seaborn-paper")

# Customize for publication quality
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# Paths
PROJECT_ROOT = Path("../..").resolve()
PROFILING_RESULTS = PROJECT_ROOT / "profiling" / "results"
BENCHMARK_RESULTS = PROJECT_ROOT / "benchmarks" / "results"
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Colors for engines
COLORS = {
    "vllm": "#2196F3",
    "sglang": "#FF5722",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Profiling results: {PROFILING_RESULTS}")
print(f"Benchmark results: {BENCHMARK_RESULTS}")

In [ ]:
"""Load profiling and benchmark data."""


def load_engine_summary(engine: str) -> dict | None:
    """Load external profiling summary for an engine."""
    summary_path = PROFILING_RESULTS / engine / "external" / "summary.json"
    if summary_path.exists():
        with open(summary_path) as f:
            return json.load(f)
    print(f"WARNING: {summary_path} not found — using placeholder data")
    return None


def load_comparison_summary() -> dict | None:
    """Load baseline comparison summary."""
    summary_path = BENCHMARK_RESULTS / "baseline" / "comparison_summary.json"
    if summary_path.exists():
        with open(summary_path) as f:
            return json.load(f)
    print(f"WARNING: {summary_path} not found — using placeholder data")
    return None


def load_request_csvs(engine: str) -> dict[int, pd.DataFrame]:
    """Load per-request CSV files for an engine."""
    result_dir = PROFILING_RESULTS / engine / "external"
    dfs = {}
    if result_dir.exists():
        for csv_file in sorted(result_dir.glob("requests_c*.csv")):
            conc = int(csv_file.stem.split("_c")[1])
            dfs[conc] = pd.read_csv(csv_file)
    return dfs


def load_gpu_csvs(engine: str) -> dict[int, pd.DataFrame]:
    """Load GPU metrics CSV files for an engine."""
    result_dir = PROFILING_RESULTS / engine / "external"
    dfs = {}
    if result_dir.exists():
        for csv_file in sorted(result_dir.glob("gpu_metrics_c*.csv")):
            conc = int(csv_file.stem.split("_c")[1])
            dfs[conc] = pd.read_csv(csv_file)
    return dfs


# Load data
vllm_summary = load_engine_summary("vllm")
sglang_summary = load_engine_summary("sglang")
comparison = load_comparison_summary()
vllm_requests = load_request_csvs("vllm")
sglang_requests = load_request_csvs("sglang")
vllm_gpu = load_gpu_csvs("vllm")
sglang_gpu = load_gpu_csvs("sglang")

# Create placeholder data if no real data exists
# TODO: Replace with actual profiling data once engines are running
if vllm_summary is None:
    print("Using placeholder data for visualization development")
    vllm_summary = {
        "1": {"throughput_tok_per_sec": 45, "ttft_p50_ms": 120, "tpot_p50_ms": 28, "gpu_utilization_mean": 35},
        "8": {"throughput_tok_per_sec": 280, "ttft_p50_ms": 180, "tpot_p50_ms": 32, "gpu_utilization_mean": 55},
        "32": {"throughput_tok_per_sec": 820, "ttft_p50_ms": 450, "tpot_p50_ms": 38, "gpu_utilization_mean": 68},
        "64": {"throughput_tok_per_sec": 1100, "ttft_p50_ms": 900, "tpot_p50_ms": 45, "gpu_utilization_mean": 72},
        "128": {"throughput_tok_per_sec": 1250, "ttft_p50_ms": 2100, "tpot_p50_ms": 55, "gpu_utilization_mean": 75},
    }
if sglang_summary is None:
    sglang_summary = {
        "1": {"throughput_tok_per_sec": 52, "ttft_p50_ms": 95, "tpot_p50_ms": 25, "gpu_utilization_mean": 40},
        "8": {"throughput_tok_per_sec": 350, "ttft_p50_ms": 140, "tpot_p50_ms": 28, "gpu_utilization_mean": 62},
        "32": {"throughput_tok_per_sec": 1050, "ttft_p50_ms": 320, "tpot_p50_ms": 32, "gpu_utilization_mean": 78},
        "64": {"throughput_tok_per_sec": 1420, "ttft_p50_ms": 650, "tpot_p50_ms": 38, "gpu_utilization_mean": 82},
        "128": {"throughput_tok_per_sec": 1610, "ttft_p50_ms": 1400, "tpot_p50_ms": 45, "gpu_utilization_mean": 84},
    }

print("Data loading complete.")

In [ ]:
"""Cell 2: Scheduling Overhead Breakdown (stacked bar chart).

Shows the breakdown of inference time into:
- Scheduling overhead (CPU)
- Model forward pass (GPU)
- KV cache management
- Other (tokenization, detokenization, data transfer)

TODO: Replace placeholder breakdown with actual profiling data from
py-spy flame graphs and cProfile analysis.
"""

# Placeholder breakdown data (TODO: derive from actual flame graphs)
concurrency_levels = [1, 8, 32, 64, 128]

# Estimated time breakdown percentages based on WukLab findings
vllm_breakdown = {
    "Scheduling": [35, 42, 50, 55, 58],
    "Model Forward": [45, 38, 30, 25, 22],
    "KV Cache Mgmt": [10, 10, 10, 10, 10],
    "Other": [10, 10, 10, 10, 10],
}

sglang_breakdown = {
    "Scheduling": [20, 25, 30, 33, 35],
    "Model Forward": [55, 50, 45, 42, 40],
    "KV Cache Mgmt": [12, 12, 12, 12, 12],
    "Other": [13, 13, 13, 13, 13],
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

x = np.arange(len(concurrency_levels))
width = 0.6
colors = ["#E53935", "#43A047", "#1E88E5", "#FDD835"]

for ax, engine_name, breakdown in [
    (axes[0], "vLLM", vllm_breakdown),
    (axes[1], "SGLang", sglang_breakdown),
]:
    bottom = np.zeros(len(concurrency_levels))
    for i, (label, values) in enumerate(breakdown.items()):
        ax.bar(x, values, width, bottom=bottom, label=label, color=colors[i], alpha=0.85)
        bottom += np.array(values)

    ax.set_xlabel("Concurrency Level")
    ax.set_xticks(x)
    ax.set_xticklabels(concurrency_levels)
    ax.set_title(f"{engine_name} — Time Breakdown")
    ax.legend(loc="upper left", framealpha=0.9)
    ax.set_ylim(0, 105)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_ylabel("% of Total Inference Time")

fig.suptitle("Scheduling Overhead Breakdown by Engine", fontsize=14, fontweight="bold")
plt.tight_layout()

fig.savefig(FIGURES_DIR / "scheduling_breakdown.png")
fig.savefig(FIGURES_DIR / "scheduling_breakdown.pdf")
plt.show()
print("Figure saved: scheduling_breakdown.{png,pdf}")

In [ ]:
"""Cell 3: Throughput vs. Concurrency (line chart).

Shows throughput (tok/s) as a function of concurrency level for both engines.
Annotates the gap percentage at each point.
"""

fig, ax = plt.subplots(figsize=(8, 5))

conc_keys = sorted(vllm_summary.keys(), key=int)
conc_vals = [int(k) for k in conc_keys]

vllm_tp = [vllm_summary[k]["throughput_tok_per_sec"] for k in conc_keys]
sglang_tp = [sglang_summary[k]["throughput_tok_per_sec"] for k in conc_keys]

ax.plot(conc_vals, vllm_tp, "o-", color=COLORS["vllm"], linewidth=2,
        markersize=8, label="vLLM", zorder=5)
ax.plot(conc_vals, sglang_tp, "s-", color=COLORS["sglang"], linewidth=2,
        markersize=8, label="SGLang", zorder=5)

# Annotate gap at each point
for i, c in enumerate(conc_vals):
    if sglang_tp[i] > 0:
        gap = ((sglang_tp[i] - vllm_tp[i]) / sglang_tp[i]) * 100
        mid_y = (vllm_tp[i] + sglang_tp[i]) / 2
        ax.annotate(
            f"{gap:.0f}%",
            xy=(c, mid_y),
            fontsize=9,
            ha="left",
            va="center",
            color="#666",
            xytext=(10, 0),
            textcoords="offset points",
        )

ax.set_xlabel("Concurrency Level")
ax.set_ylabel("Throughput (tokens/sec)")
ax.set_title("Throughput vs. Concurrency: vLLM vs SGLang")
ax.legend()
ax.set_xscale("log", base=2)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "throughput_vs_concurrency.png")
fig.savefig(FIGURES_DIR / "throughput_vs_concurrency.pdf")
plt.show()
print("Figure saved: throughput_vs_concurrency.{png,pdf}")

In [ ]:
"""Cell 4: Latency CDFs (line chart).

TTFT and TPOT cumulative distribution functions at concurrency=32
(or closest available) for each engine.
"""

TARGET_CONC = 32

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Try to load actual per-request data; fall back to synthetic CDFs
has_data = bool(vllm_requests) and bool(sglang_requests)

if has_data:
    # Find closest concurrency level with data
    vllm_conc = min(vllm_requests.keys(), key=lambda c: abs(c - TARGET_CONC))
    sglang_conc = min(sglang_requests.keys(), key=lambda c: abs(c - TARGET_CONC))
    vllm_df = vllm_requests[vllm_conc]
    sglang_df = sglang_requests[sglang_conc]
else:
    # Generate synthetic CDF data for visualization development
    # TODO: Replace with actual data
    np.random.seed(42)
    vllm_df = pd.DataFrame({
        "ttft_ms": np.random.lognormal(mean=5.5, sigma=0.8, size=200),
        "tpot_ms": np.random.lognormal(mean=3.5, sigma=0.5, size=200),
    })
    sglang_df = pd.DataFrame({
        "ttft_ms": np.random.lognormal(mean=5.2, sigma=0.6, size=200),
        "tpot_ms": np.random.lognormal(mean=3.3, sigma=0.4, size=200),
    })

# TTFT CDF
for engine, df, color in [("vLLM", vllm_df, COLORS["vllm"]),
                            ("SGLang", sglang_df, COLORS["sglang"])]:
    sorted_vals = np.sort(df["ttft_ms"].dropna())
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    axes[0].plot(sorted_vals, cdf, color=color, linewidth=2, label=engine)

axes[0].set_xlabel("TTFT (ms)")
axes[0].set_ylabel("CDF")
axes[0].set_title(f"Time to First Token CDF (concurrency~{TARGET_CONC})")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0.50, color="gray", linestyle="--", alpha=0.5, label="p50")
axes[0].axhline(y=0.99, color="gray", linestyle=":", alpha=0.5, label="p99")

# TPOT CDF
for engine, df, color in [("vLLM", vllm_df, COLORS["vllm"]),
                            ("SGLang", sglang_df, COLORS["sglang"])]:
    tpot_vals = df["tpot_ms"].dropna()
    tpot_vals = tpot_vals[tpot_vals > 0]
    if len(tpot_vals) > 0:
        sorted_vals = np.sort(tpot_vals)
        cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
        axes[1].plot(sorted_vals, cdf, color=color, linewidth=2, label=engine)

axes[1].set_xlabel("TPOT (ms)")
axes[1].set_ylabel("CDF")
axes[1].set_title(f"Time per Output Token CDF (concurrency~{TARGET_CONC})")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0.50, color="gray", linestyle="--", alpha=0.5)
axes[1].axhline(y=0.99, color="gray", linestyle=":", alpha=0.5)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "latency_cdfs.png")
fig.savefig(FIGURES_DIR / "latency_cdfs.pdf")
plt.show()
print("Figure saved: latency_cdfs.{png,pdf}")

In [ ]:
"""Cell 5: GPU Utilization Time Series.

GPU utilization % over time during a benchmark run,
overlaid for both engines.
"""

fig, ax = plt.subplots(figsize=(10, 4))

has_gpu_data = bool(vllm_gpu) and bool(sglang_gpu)

if has_gpu_data:
    # Use data from the highest concurrency level available
    vllm_conc = max(vllm_gpu.keys())
    sglang_conc = max(sglang_gpu.keys())

    for engine, gpu_df, color, conc in [
        ("vLLM", vllm_gpu[vllm_conc], COLORS["vllm"], vllm_conc),
        ("SGLang", sglang_gpu[sglang_conc], COLORS["sglang"], sglang_conc),
    ]:
        if "timestamp" in gpu_df.columns and "utilization_pct" in gpu_df.columns:
            t = gpu_df["timestamp"] - gpu_df["timestamp"].iloc[0]
            ax.plot(t, gpu_df["utilization_pct"], color=color, alpha=0.7,
                    linewidth=1.5, label=f"{engine} (c={conc})")
else:
    # Synthetic GPU utilization data for visualization development
    # TODO: Replace with actual data
    np.random.seed(42)
    t = np.linspace(0, 60, 600)

    # vLLM: lower, more variable utilization
    vllm_util = 65 + 15 * np.sin(t * 0.3) + np.random.normal(0, 5, len(t))
    vllm_util = np.clip(vllm_util, 0, 100)

    # SGLang: higher, more stable utilization
    sglang_util = 80 + 8 * np.sin(t * 0.2) + np.random.normal(0, 3, len(t))
    sglang_util = np.clip(sglang_util, 0, 100)

    ax.plot(t, vllm_util, color=COLORS["vllm"], alpha=0.7,
            linewidth=1.5, label="vLLM (c=128)")
    ax.plot(t, sglang_util, color=COLORS["sglang"], alpha=0.7,
            linewidth=1.5, label="SGLang (c=128)")

ax.set_xlabel("Time (seconds)")
ax.set_ylabel("GPU Utilization (%)")
ax.set_title("GPU Utilization During Benchmark")
ax.legend()
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "gpu_utilization_timeseries.png")
fig.savefig(FIGURES_DIR / "gpu_utilization_timeseries.pdf")
plt.show()
print("Figure saved: gpu_utilization_timeseries.{png,pdf}")

## Flame Graph Analysis

TODO: Run the following to generate flame graphs, then analyze below:
```bash
python profiling/scripts/profile_vllm_scheduler.py --profile-internal
python profiling/scripts/profile_sglang_scheduler.py --profile-internal
```

### vLLM Flame Graph
Expected location: `profiling/results/vllm/internal/vllm_flamegraph.svg`

**Key scheduling-related functions to look for:**
- `Scheduler.schedule()` — main scheduling entry point
- `Scheduler._schedule_default()` — default scheduling policy
- `BlockSpaceManager.can_allocate()` — KV cache allocation check
- `BlockSpaceManager.allocate()` — actual block allocation
- `_process_model_outputs()` — output processing (detokenization)
- `_prepare_model_input()` — input tensor preparation

### SGLang Flame Graph
Expected location: `profiling/results/sglang/internal/sglang_flamegraph.svg`

**Key functions to look for:**
- RadixAttention tree operations (insert, match, evict)
- Batch scheduler functions (schedule_batch, update_running)
- Tokenizer pipeline (encode, decode)
- FlashInfer kernel dispatch overhead

### Expected Findings
Based on WukLab's study, we expect:
1. vLLM spends >50% of time in CPU scheduling at high concurrency
2. `Scheduler.schedule()` and `_prepare_model_input()` dominate
3. SGLang's scheduling is lighter due to RadixAttention's shared prefix handling
4. The gap widens at higher concurrency levels

## Key Findings Summary

TODO: Fill in with actual measurements after running profiling.

### 1. Scheduling Overhead
- **vLLM:** Scheduling consumes ~XX% of total inference time at concurrency 32+
- **SGLang:** Scheduling consumes ~XX% of total inference time at concurrency 32+
- The gap grows with concurrency, confirming WukLab's findings

### 2. Throughput Gap
- SGLang outperforms vLLM by ~XX% in throughput across concurrency levels
- Gap is most pronounced at concurrency XX (XX% difference)
- Consistent with the reported ~29% gap using identical FlashInfer kernels

### 3. Top Bottleneck Functions (from flame graphs)
1. TODO: `function_name()` — XX% of total time
2. TODO: `function_name()` — XX% of total time
3. TODO: `function_name()` — XX% of total time

### 4. Intervention Points for WorkloadRouter
Based on profiling, the WorkloadRouter should target:
1. **Batch construction:** Pre-sorting requests by expected length reduces scheduling decisions
2. **KV cache pre-allocation:** Predicting cache needs at routing time avoids scheduling-time allocation checks
3. **Request grouping:** Length-aware grouping minimizes padding waste and batch heterogeneity
4. **Async scheduling:** Overlapping scheduling with GPU execution (pipeline scheduling)